In [5]:
#Leitura de arquivo;

path = ("/Workspace/Users/brunoagra988@gmail.com/databricks_analysis_project/data/EmendasParlamentares.csv")
df_emendas = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";") 
    .option("encoding", "ISO-8859-1")  
    .load("/Workspace/Users/brunoagra988@gmail.com/databricks_analysis_project/data/EmendasParlamentares.csv")
)

In [6]:
# Limpeza de nomes, colunas e também adição da coluna de data de carga para rastreabilidade.

import re
from pyspark.sql.functions import current_timestamp

def limpar_nome_coluna(nome):
    nome_limpo = re.sub(r'[ ,;{}()\n\t=]', '_', nome)
    nome_limpo = re.sub(r'_+', '_', nome_limpo)
    return nome_limpo.strip('_')

colunas_limpas = [limpar_nome_coluna(c) for c in df_emendas.columns]

df_emendas_limpo = df_emendas.toDF(*colunas_limpas)

df_emendas_limpo = df_emendas_limpo.withColumn(
    "Data_Carga",
    current_timestamp()
)

tabela = "emendas_parlamentares_bronze"

if "Data_Carga" not in spark.table(tabela).columns:
    spark.sql(f"ALTER TABLE {tabela} ADD COLUMNS (Data_Carga TIMESTAMP)")

df_emendas_limpo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabela)